# NGIF Tracer Particles

This single notebook generates tracer snapshots, trains `NGIF-div`, `NGIF-curl`, `NGIF-kin`, and the gradient baseline, rolls all trained models out in memory, and saves the comparison animation directly to `./scatter_movie_grid.gif`. No intermediate files are written except the final GIF.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
from IPython.display import HTML

from ngif.data import TracerConfig, generate_tracer_data, normalize_snapshots, x_from_unit
from ngif.rff import prepare_rff_features
from ngif.rollout import histogram_tv, rollout_model
from ngif.plot import plot_movie_grid, scatter_movie_grid
from ngif.train import TrainConfig, train_all_variants


In [ ]:
# Increase these values for smoother trajectories and better fits.
n_particles = 512
n_steps = 33
train_steps = 10_000
batch_size = 128
n_frequencies = 256
bandwidths = (0.08, 0.2, 0.5, 1.0)
variants = ("div", "curl", "kin", "grad")
titles = ["data", "NGIF-div", "NGIF-curl", "NGIF-kin", "grad"]
gauge_weights = {"div": 1e-2, "curl": 1e-2, "kin": 1e-2, "grad": 0.0}
output_path = Path("./scatter_movie_grid.gif")


In [ ]:
data_config = TracerConfig(
    n_particles=n_particles,
    n_steps=n_steps,
    t_final=3.0,
    seed=3,
    substeps_per_frame=12,
)
data = generate_tracer_data(data_config)
x_norm, t_norm, norm_info = normalize_snapshots(data["x"], data["t"])

features = prepare_rff_features(
    x_norm,
    t_norm,
    key=0,
    n_frequencies=n_frequencies,
    bandwidths=bandwidths,
    spline_lam=1e-5,
)

print("snapshots", x_norm.shape)
print("RFF omega", features["omega"].shape)
print("moment derivatives", features["moment_derivatives"].shape)


In [ ]:
base_train_config = TrainConfig(
    steps=train_steps,
    batch_size=batch_size,
    learning_rate=5e-4,
    gauge_weight=1e-2,
    width=64,
    depth=4,
    seed=10,
    log_every=max(train_steps // 30, 1),
    verbose=True,
)

results = train_all_variants(
    x_norm,
    t_norm,
    features["omega"],
    features["moment_derivatives"],
    base_train_config,
    variants=variants,
    gauge_weights=gauge_weights,
)


In [ ]:
fig, ax = plt.subplots(figsize=(7, 4))
for variant, result in results.items():
    history = result["history"]
    ax.semilogy(history["step"], history["loss"], label=variant)
ax.set_xlabel("optimizer step")
ax.set_ylabel("training loss")
ax.legend()
fig.tight_layout()


In [ ]:
rollouts_norm = {}
for variant, result in results.items():
    rollouts_norm[variant] = rollout_model(
        result["model"],
        result["params"],
        variant,
        x_norm[0],
        t_norm,
        substeps=4,
    )
    print(variant, rollouts_norm[variant].shape)


In [ ]:
fig, ax = plt.subplots(figsize=(7, 4))
for variant, rollout in rollouts_norm.items():
    tv = histogram_tv(x_norm, rollout, bins=36)
    ax.plot(data["t"], tv, label=variant)
ax.set_xlabel("time")
ax.set_ylabel("histogram TV distance")
ax.legend()
fig.tight_layout()


In [ ]:
panels_norm = np.stack([x_norm, *(rollouts_norm[v] for v in variants)], axis=0)
panels_phys = x_from_unit(panels_norm, domain_size=norm_info["domain_size"])

anim = scatter_movie_grid(
    panels_phys,
    t=data["t"],
    titles_x=titles,
    grid_width=3,
    fig_size=(11, 7),
    frames=40,
    n_samples=220,
    n_traj=50,
    xlim=(0, 2 * np.pi),
    ylim=(0, 2 * np.pi),
    plot_trajectories=True,
    periodic=True,
    trajectory_length=10,
    save_to=output_path,
    show=False,
)


In [ ]:
hist_output_path = Path("./hist_movie_grid.gif")

CMAP = "mako"
HIST_RES = 256
SQ_RT = 0.7

def get_hist(frame, hist_res=256, range=((0, np.pi * 2), (0, np.pi * 2))):
    H, _, _ = np.histogram2d(frame[..., 0], frame[..., 1], bins=hist_res, range=range)
    return H.T

Hs = [[get_hist(s, hist_res=HIST_RES) for s in ss] for ss in panels_phys]
Hs = np.stack(Hs)
Hs = np.asarray(Hs) / 5
Hs = np.clip(Hs, 0, 1.0)
Hs = np.asarray(Hs) ** SQ_RT
Hs = np.concatenate(Hs)
print(Hs.shape)

hist_anim = plot_movie_grid(
    Hs,
    n_movies=len(titles),
    t=data["t"],
    titles_x=titles,
    grid_width=3,
    fig_size=(11, 7),
    frames=40,
    cmap=CMAP,
    c_norm=(0, 1),
    save_to=hist_output_path,
    show=False,
)
HTML(hist_anim.to_jshtml())
